# 01 - Exploration des donnees IEEE-CIS Fraud Detection

Objectif de ce notebook : construire une premiere comprehension metier et statistique du dataset, puis demontrer pourquoi l'accuracy est une metrique dangereuse dans un probleme de fraude fortement desequilibre.

Ce notebook couvre :
- chargement des donnees transactionnelles et identite ;
- verification du desequilibre de classes ;
- analyse temporelle ;
- analyse des montants ;
- valeurs manquantes et variables categorielles ;
- correlations simples avec la cible ;
- baseline naive avec regression logistique ;
- premiers exemples de features temporelles sans fuite de donnees.

## 0. Installation et imports

Si une librairie manque dans ton environnement, installe-la avant de relancer le notebook. Exemple :

```bash
pip install pandas numpy matplotlib seaborn scikit-learn
```

Le notebook ne telecharge pas les donnees : il suppose que les CSV IEEE-CIS sont deja dans `../data/`.

In [ ]:
from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    recall_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

# Le chemin depend du dossier depuis lequel le kernel Jupyter est lance.
# On supporte les deux cas courants : racine du projet ou dossier notebooks/.
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
TRAIN_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
TRAIN_IDENTITY_PATH = DATA_DIR / "train_identity.csv"
TEST_TRANSACTION_PATH = DATA_DIR / "test_transaction.csv"
TEST_IDENTITY_PATH = DATA_DIR / "test_identity.csv"

for path in [TRAIN_TRANSACTION_PATH, TRAIN_IDENTITY_PATH, TEST_TRANSACTION_PATH, TEST_IDENTITY_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path.resolve()}")

## 1. Chargement des donnees

Le dataset IEEE-CIS separe les transactions et les informations d'identite. On fusionne les deux tables sur `TransactionID` avec une jointure gauche : toutes les transactions sont conservees, meme quand les informations d'identite sont absentes.

Important : la colonne `TransactionDT` est un temps relatif, pas une date calendaire reelle. On l'utilise comme axe temporel ordonne.

In [ ]:
start = time.time()

train_transaction = pd.read_csv(TRAIN_TRANSACTION_PATH)
train_identity = pd.read_csv(TRAIN_IDENTITY_PATH)

train = train_transaction.merge(train_identity, on="TransactionID", how="left")

print(f"train_transaction : {train_transaction.shape}")
print(f"train_identity    : {train_identity.shape}")
print(f"train fusionne    : {train.shape}")
print(f"Temps de chargement : {time.time() - start:.1f} secondes")

# On libere les tables intermediaires pour reduire l'empreinte memoire.
del train_transaction, train_identity
gc.collect()

train.head()

In [ ]:
# Verification rapide des types de variables et des colonnes principales.
print(train.info())
train[["TransactionID", "isFraud", "TransactionDT", "TransactionAmt", "ProductCD"]].head()

## 2. Desequilibre de classes

En detection de fraude, la classe positive est rare. Une accuracy elevee peut donc simplement signifier que le modele predit presque toujours "non fraude". On commence par mesurer ce desequilibre.

In [ ]:
target_counts = train["isFraud"].value_counts().rename(index={0: "legitime", 1: "fraude"})
target_rates = train["isFraud"].value_counts(normalize=True).rename(index={0: "legitime", 1: "fraude"})

display(pd.DataFrame({"count": target_counts, "rate": target_rates}))

baseline_accuracy = 1 - train["isFraud"].mean()
print(f"Accuracy d'un classifieur qui predit toujours 'legitime' : {baseline_accuracy:.4%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=train, x="isFraud", ax=ax)
ax.set_title("Distribution de la cible")
ax.set_xlabel("isFraud")
ax.set_ylabel("Nombre de transactions")
ax.set_xticklabels(["Legitime", "Fraude"])
plt.show()

Conclusion attendue : l'accuracy seule n'est pas adaptee. Les metriques importantes pour la suite sont plutot :
- **Recall fraude** : quelle part des fraudes est detectee ?
- **Precision** : quelle part des alertes sont vraiment frauduleuses ?
- **F1-score** : compromis precision / recall a un seuil donne.
- **AUPRC** : aire sous la courbe precision-recall, plus informative que l'AUROC en cas de fort desequilibre.

## 3. Analyse temporelle

`TransactionDT` est un timestamp relatif en secondes. On cree des variables interpretables : heure relative, jour relatif et semaine relative. Ces variables ne doivent pas etre melangees aleatoirement dans les splits, car cela ferait apprendre au modele des informations futures.

In [ ]:
SECONDS_PER_HOUR = 3600
SECONDS_PER_DAY = 24 * SECONDS_PER_HOUR
SECONDS_PER_WEEK = 7 * SECONDS_PER_DAY

train = train.sort_values("TransactionDT").reset_index(drop=True)
train["hour_rel"] = (train["TransactionDT"] // SECONDS_PER_HOUR).astype(int)
train["day_rel"] = (train["TransactionDT"] // SECONDS_PER_DAY).astype(int)
train["week_rel"] = (train["TransactionDT"] // SECONDS_PER_WEEK).astype(int)
train["hour_of_day_rel"] = ((train["TransactionDT"] // SECONDS_PER_HOUR) % 24).astype(int)

train[["TransactionDT", "hour_rel", "day_rel", "week_rel", "hour_of_day_rel"]].head()

In [ ]:
daily = (
    train.groupby("day_rel")
    .agg(
        transactions=("TransactionID", "count"),
        frauds=("isFraud", "sum"),
        fraud_rate=("isFraud", "mean"),
        amount_mean=("TransactionAmt", "mean"),
    )
    .reset_index()
)

display(daily.head())

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
sns.lineplot(data=daily, x="day_rel", y="transactions", ax=axes[0])
axes[0].set_title("Volume quotidien de transactions")
axes[0].set_ylabel("Transactions")

sns.lineplot(data=daily, x="day_rel", y="fraud_rate", ax=axes[1], color="crimson")
axes[1].set_title("Taux de fraude quotidien")
axes[1].set_ylabel("Taux de fraude")
axes[1].set_xlabel("Jour relatif")
plt.tight_layout()
plt.show()

In [ ]:
hourly_rate = train.groupby("hour_of_day_rel")["isFraud"].agg(["count", "mean"]).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 4))
sns.barplot(data=hourly_rate, x="hour_of_day_rel", y="count", color="lightsteelblue", ax=ax1)
ax1.set_ylabel("Nombre de transactions")
ax1.set_xlabel("Heure relative dans la journee")

ax2 = ax1.twinx()
sns.lineplot(data=hourly_rate, x="hour_of_day_rel", y="mean", color="crimson", marker="o", ax=ax2)
ax2.set_ylabel("Taux de fraude")
ax1.set_title("Volume et taux de fraude par heure relative")
plt.show()

display(hourly_rate.rename(columns={"count": "transactions", "mean": "fraud_rate"}))

## 4. Analyse des montants

Les montants sont souvent tres asymetriques. On affiche donc `TransactionAmt` en echelle logarithmique pour rendre les petites et moyennes transactions visibles.

In [ ]:
amount_summary = train.groupby("isFraud")["TransactionAmt"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
display(amount_summary.rename(index={0: "legitime", 1: "fraude"}))

In [ ]:
plot_df = train[["TransactionAmt", "isFraud"]].copy()
plot_df["log_amount"] = np.log1p(plot_df["TransactionAmt"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=plot_df, x="log_amount", hue="isFraud", bins=80, stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Distribution log(1 + montant)")
axes[0].set_xlabel("log(1 + TransactionAmt)")

sns.boxplot(data=train, x="isFraud", y="TransactionAmt", showfliers=False, ax=axes[1])
axes[1].set_title("Montants par classe, sans outliers visibles")
axes[1].set_xlabel("isFraud")
axes[1].set_ylabel("TransactionAmt")
axes[1].set_xticklabels(["Legitime", "Fraude"])
plt.tight_layout()
plt.show()

## 5. Variables categorielles importantes

On regarde quelques variables metier lisibles : produit, type de carte, domaines email, type d'appareil. Les taux par categorie doivent etre interpretes avec prudence si le nombre d'observations est faible.

In [ ]:
def fraud_rate_by_category(df, col, min_count=500, top_n=20):
    """Resume le volume et le taux de fraude par modalite."""
    out = (
        df.groupby(col, dropna=False)
        .agg(transactions=("TransactionID", "count"), fraud_rate=("isFraud", "mean"), frauds=("isFraud", "sum"))
        .query("transactions >= @min_count")
        .sort_values("fraud_rate", ascending=False)
        .head(top_n)
    )
    return out

categorical_cols_to_check = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain", "DeviceType"]

for col in categorical_cols_to_check:
    if col in train.columns:
        print(f"\n=== {col} ===")
        display(fraud_rate_by_category(train, col))

## 6. Valeurs manquantes

Les valeurs manquantes sont nombreuses dans IEEE-CIS. Elles peuvent etre informatives : l'absence d'information d'identite ou device peut elle-meme correspondre a un pattern de risque.

In [ ]:
missing = (
    train.isna().mean()
    .sort_values(ascending=False)
    .to_frame("missing_rate")
)
missing["missing_count"] = train.isna().sum().loc[missing.index]
display(missing.head(30))

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(missing["missing_rate"], bins=30, ax=ax)
ax.set_title("Distribution des taux de valeurs manquantes par colonne")
ax.set_xlabel("Taux de valeurs manquantes")
plt.show()

## 7. Correlations simples avec la cible

Cette analyse donne un premier signal, mais elle ne remplace pas un modele. Une faible correlation lineaire ne veut pas dire qu'une variable est inutile : les relations peuvent etre non lineaires ou dependantes d'interactions.

In [ ]:
numeric_cols = train.select_dtypes(include=[np.number]).columns.drop(["isFraud"], errors="ignore")

# Pour accelerer le calcul, on ignore les colonnes presque entierement manquantes.
usable_numeric_cols = [col for col in numeric_cols if train[col].notna().mean() >= 0.2]

corr_with_target = (
    train[usable_numeric_cols + ["isFraud"]]
    .corr(numeric_only=True)["isFraud"]
    .drop("isFraud")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

display(corr_with_target.head(25).to_frame("corr_with_isFraud"))

fig, ax = plt.subplots(figsize=(8, 7))
corr_with_target.head(20).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top correlations absolues avec isFraud")
ax.set_xlabel("Correlation lineaire")
plt.show()

## 8. Split temporel et baseline naive

On utilise un split temporel : les premieres transactions servent a entrainer, les dernieres a tester. C'est essentiel car en production le modele est toujours applique au futur.

Un `KFold` classique melangerait passe et futur. Cela introduirait une fuite temporelle : le modele serait evalue dans un contexte plus facile que la vraie production.

In [ ]:
# Colonnes simples pour une baseline volontairement naive.
# On evite les centaines de colonnes V pour garder un premier modele lisible et rapide.
baseline_features = [
    "TransactionAmt",
    "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2",
    "dist1", "dist2",
    "P_emaildomain", "R_emaildomain",
    "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13", "C14",
    "D1", "D2", "D3", "D4", "D5", "D10", "D15",
    "hour_of_day_rel", "day_rel", "week_rel",
    "DeviceType",
]
baseline_features = [col for col in baseline_features if col in train.columns]

X = train[baseline_features]
y = train["isFraud"].astype(int)

split_idx = int(len(train) * 0.8)
X_train, X_valid = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_valid = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train : {X_train.shape}, periode jours {train['day_rel'].iloc[:split_idx].min()} -> {train['day_rel'].iloc[:split_idx].max()}")
print(f"Valid : {X_valid.shape}, periode jours {train['day_rel'].iloc[split_idx:].min()} -> {train['day_rel'].iloc[split_idx:].max()}")
print(f"Taux fraude train : {y_train.mean():.4%}")
print(f"Taux fraude valid : {y_valid.mean():.4%}")

In [ ]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

# Modele volontairement simple, sans traitement du desequilibre.
baseline_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=500, n_jobs=-1, random_state=RANDOM_STATE)),
    ]
)

start = time.time()
baseline_model.fit(X_train, y_train)
fit_time = time.time() - start
print(f"Temps d'entrainement baseline : {fit_time:.1f} secondes")

In [ ]:
valid_scores = baseline_model.predict_proba(X_valid)[:, 1]
valid_pred = (valid_scores >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_valid, valid_pred),
    "recall_fraude": recall_score(y_valid, valid_pred, zero_division=0),
    "f1": f1_score(y_valid, valid_pred, zero_division=0),
    "AUPRC": average_precision_score(y_valid, valid_scores),
}

display(pd.Series(metrics).to_frame("baseline_logistic"))
print(classification_report(y_valid, valid_pred, target_names=["legitime", "fraude"], zero_division=0))

cm = confusion_matrix(y_valid, valid_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Matrice de confusion - baseline logistique")
ax.set_xlabel("Prediction")
ax.set_ylabel("Vrai label")
ax.set_xticklabels(["Legitime", "Fraude"])
ax.set_yticklabels(["Legitime", "Fraude"], rotation=0)
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_valid, valid_scores)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, label=f"LogReg AUPRC = {metrics['AUPRC']:.4f}")
ax.axhline(y_valid.mean(), color="gray", linestyle="--", label=f"Prevalence fraude = {y_valid.mean():.4f}")
ax.set_title("Courbe Precision-Recall")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.show()

### Interpretation de la baseline

A completer apres execution :

- L'accuracy est-elle elevee ?
- Le recall sur la classe fraude est-il satisfaisant ?
- L'AUPRC est-elle nettement superieure au taux de fraude de validation ?
- Que se passerait-il metier si ce modele etait deploye tel quel ?

Conclusion probable : l'accuracy donne une impression de performance, mais le modele peut manquer trop de fraudes. Pour PayTrack, les faux negatifs sont couteux, donc le recall et l'AUPRC doivent guider les comparaisons.

## 9. Demonstration de TimeSeriesSplit

La suite du projet doit utiliser des validations temporelles. Cette cellule montre la logique des folds : chaque validation est posterieure a son entrainement.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

folds = []
for fold, (train_idx, valid_idx) in enumerate(tscv.split(train), start=1):
    folds.append(
        {
            "fold": fold,
            "train_rows": len(train_idx),
            "valid_rows": len(valid_idx),
            "train_day_min": train.loc[train_idx, "day_rel"].min(),
            "train_day_max": train.loc[train_idx, "day_rel"].max(),
            "valid_day_min": train.loc[valid_idx, "day_rel"].min(),
            "valid_day_max": train.loc[valid_idx, "day_rel"].max(),
            "valid_fraud_rate": train.loc[valid_idx, "isFraud"].mean(),
        }
    )

display(pd.DataFrame(folds))

## 10. Premier feature engineering temporel

Les variables les plus utiles en fraude sont souvent des ecarts au comportement habituel. Ici on cree des exemples simples en respectant l'ordre temporel :
- nombre de transactions precedentes pour une carte ;
- montant moyen historique precedent ;
- ratio entre montant actuel et moyenne historique precedente ;
- indicateur de nouveau marchand pour une carte.

Remarque : dans IEEE-CIS, il n'y a pas de vrai identifiant client explicite. Pour ce POC, on approxime le client par une combinaison de champs carte/adresse. Cette hypothese doit etre documentee dans le rendu.

In [ ]:
def add_temporal_behavior_features(df):
    """Cree des features historiques sans utiliser la transaction courante dans l'historique."""
    df = df.sort_values("TransactionDT").copy()

    # Identifiant client approximatif : suffisamment stable pour l'exploration,
    # mais a discuter dans le rapport car ce n'est pas un vrai customer_id.
    id_cols = [col for col in ["card1", "card2", "card3", "card5", "addr1"] if col in df.columns]
    df["customer_proxy"] = df[id_cols].astype("string").fillna("missing").agg("_".join, axis=1)

    grouped = df.groupby("customer_proxy", sort=False)

    # Nombre de transactions deja vues pour ce proxy client avant la transaction courante.
    df["customer_tx_count_prev"] = grouped.cumcount()

    # Moyenne historique precedente du montant : shift pour exclure la transaction courante.
    df["customer_amt_mean_prev"] = grouped["TransactionAmt"].transform(lambda s: s.shift(1).expanding().mean())
    df["amt_to_customer_mean_prev"] = df["TransactionAmt"] / df["customer_amt_mean_prev"].replace(0, np.nan)

    # Nouveau marchand approxime par ProductCD + email marchand receveur si disponible.
    merchant_cols = [col for col in ["ProductCD", "R_emaildomain"] if col in df.columns]
    df["merchant_proxy"] = df[merchant_cols].astype("string").fillna("missing").agg("_".join, axis=1)
    df["merchant_seen_before"] = (
        df.groupby(["customer_proxy", "merchant_proxy"], sort=False).cumcount() > 0
    ).astype(int)
    df["is_new_merchant_for_customer"] = 1 - df["merchant_seen_before"]

    return df

train_fe = add_temporal_behavior_features(train)
new_features = [
    "customer_tx_count_prev",
    "customer_amt_mean_prev",
    "amt_to_customer_mean_prev",
    "is_new_merchant_for_customer",
]
display(train_fe[new_features + ["isFraud"]].head(10))
display(train_fe.groupby("isFraud")[new_features].describe().T.head(40))

### Limite importante

Les features de fenetres exactes 1h / 24h / 7j peuvent etre ajoutees ensuite, mais elles demandent une implementation plus couteuse. Pour la partie modelisation, il faudra idealement les calculer avec des `rolling` temporels par `customer_proxy`, en s'assurant que la transaction courante est exclue de la fenetre historique.

Regle a conserver pour tout feature engineering temporel : **une ligne ne doit jamais utiliser une information posterieure a `TransactionDT`**.

## 11. Synthese EDA a reporter

A completer apres execution du notebook :

1. Taux de fraude observe : `...`.
2. Accuracy du classifieur majoritaire : `...`.
3. Taux de fraude variable dans le temps : oui / non, observations principales : `...`.
4. Variables ou familles de variables prometteuses : `...`.
5. Baseline logistique : accuracy `...`, recall fraude `...`, AUPRC `...`.
6. Conclusion metier : l'accuracy ne suffit pas ; la suite doit optimiser AUPRC et recall sous contrainte de precision pour ne pas saturer les analystes.